# DocLite: Knowledge Distillation for Document Understanding

**Teacher:** LayoutLMv3 (text + layout + image) → **Student:** LiLT (text + layout only)

This notebook runs all experiments:
1. LayoutLMv3 supervised baseline (teacher upper bound)
2. LiLT supervised baseline (student lower bound)
3. DocLite distillation (our contribution)
4. Multimodal distillation with images

On both **FUNSD** (forms) and **SROIE** (receipts) datasets.

## 0. Setup (Run this first)

In [ ]:
# Clone repo and install dependencies
!git clone https://github.com/lukeengel/doclite.git
%cd doclite
!pip install -r requirements.txt -q
!apt-get install -y tesseract-ocr -qq

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Upload SROIE dataset (optional)

If you have SROIE, zip it and upload to Google Drive, then run the cell below.
If you only want FUNSD experiments, skip this.

In [ ]:
# Uncomment to mount Google Drive and copy SROIE data
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip /content/drive/MyDrive/sroie.zip -d data/

## 1. Imports & Data Loading

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, LayoutLMv3Config, LayoutLMv3ForTokenClassification
from collections import Counter

from doclite.configs.core import ENV, WEIGHTS
from doclite.data.document_dataset import DocumentDataset
from doclite.data.collate import collate_document_batch
from doclite.eval.evaluate import evaluate_student
from doclite.models.teacher import TeacherModel
from doclite.models.student import StudentModel
from doclite.distill.distill_loss import compute_distill_loss
from build_funsd_examples import load_funsd_split

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Hyperparameters
BATCH_SIZE = 8  # Increase on GPU (vs 2 on CPU)
NUM_EPOCHS = 5
LR = 5e-5
NUM_LABELS_FUNSD = 4  # question, answer, header, other

print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Learning rate: {LR}")

---
# PART 1: FUNSD Experiments (Text + Layout)
---

## 2. Load FUNSD Data

In [ ]:
FUNSD_ROOT = ENV.DATA / "funsd"
TRAIN_ANN_DIR = FUNSD_ROOT / "training_data" / "annotations"
TEST_ANN_DIR = FUNSD_ROOT / "testing_data" / "annotations"

tokenizer = AutoTokenizer.from_pretrained("microsoft/layoutlmv3-base")

funsd_train = load_funsd_split(TRAIN_ANN_DIR, tokenizer)
funsd_test = load_funsd_split(TEST_ANN_DIR, tokenizer)

funsd_train_loader = DataLoader(
    DocumentDataset(funsd_train), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_document_batch
)
funsd_test_loader = DataLoader(
    DocumentDataset(funsd_test), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_document_batch
)

print(f"FUNSD — Train: {len(funsd_train)} docs ({len(funsd_train_loader)} batches) | Test: {len(funsd_test)} docs")

# Label distribution
ID2LABEL_FUNSD = {0: "question", 1: "answer", 2: "header", 3: "other", -100: "[PAD]"}
all_labels = [l for ex in funsd_train for l in ex["labels"] if l != -100]
counts = Counter(all_labels)
print("\nLabel distribution (training):")
for lid, c in sorted(counts.items()):
    print(f"  {ID2LABEL_FUNSD[lid]:>10s}: {c:>5d} ({c/len(all_labels)*100:.1f}%)")

## 3. Experiment A: LayoutLMv3 Baseline (Teacher Upper Bound)

In [ ]:
config = LayoutLMv3Config.from_pretrained(
    "microsoft/layoutlmv3-base", num_labels=NUM_LABELS_FUNSD,
    output_hidden_states=True, output_attentions=True,
)
teacher_model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base", config=config
).to(device)

optimizer_t = torch.optim.AdamW(teacher_model.parameters(), lr=LR, weight_decay=0.01)

class LayoutLMv3EvalWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids, attention_mask, bbox, **kwargs):
        fwd = dict(input_ids=input_ids, attention_mask=attention_mask, bbox=bbox)
        if 'pixel_values' in kwargs:
            fwd['pixel_values'] = kwargs['pixel_values']
        return {"logits": self.model(**fwd).logits}

eval_wrapper_t = LayoutLMv3EvalWrapper(teacher_model).to(device)
print(f"LayoutLMv3 parameters: {sum(p.numel() for p in teacher_model.parameters()):,}")

In [ ]:
funsd_teacher_results = []

for epoch in range(NUM_EPOCHS):
    teacher_model.train()
    total_loss = 0.0

    for step, batch in enumerate(funsd_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer_t.zero_grad()

        out = teacher_model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
        )
        out.loss.backward()
        optimizer_t.step()
        total_loss += out.loss.item()

        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(funsd_train_loader)} | loss={out.loss.item():.4f}")

    metrics = evaluate_student(eval_wrapper_t, funsd_test_loader, device=device)
    funsd_teacher_results.append(metrics)
    print(f"Epoch {epoch+1} | train_loss={total_loss/len(funsd_train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

print(f"\n>>> LayoutLMv3 Best F1: {max(r['token_f1'] for r in funsd_teacher_results):.4f}")

## 4. Experiment B: LiLT Baseline (Student Lower Bound)

In [ ]:
student_baseline = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_FUNSD).to(device)
optimizer_sb = torch.optim.AdamW(student_baseline.parameters(), lr=LR, weight_decay=0.01)

print(f"LiLT parameters: {sum(p.numel() for p in student_baseline.parameters()):,}")

In [ ]:
funsd_student_results = []

for epoch in range(NUM_EPOCHS):
    student_baseline.train()
    total_loss = 0.0

    for step, batch in enumerate(funsd_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer_sb.zero_grad()

        out = student_baseline.model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
        )
        out.loss.backward()
        optimizer_sb.step()
        total_loss += out.loss.item()

        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(funsd_train_loader)} | loss={out.loss.item():.4f}")

    metrics = evaluate_student(student_baseline, funsd_test_loader, device=device)
    funsd_student_results.append(metrics)
    print(f"Epoch {epoch+1} | train_loss={total_loss/len(funsd_train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

print(f"\n>>> LiLT Baseline Best F1: {max(r['token_f1'] for r in funsd_student_results):.4f}")

## 5. Experiment C: DocLite Distillation (Text + Layout)

In [ ]:
teacher = TeacherModel("microsoft/layoutlmv3-base", num_labels=NUM_LABELS_FUNSD).to(device)
for param in teacher.parameters():
    param.requires_grad = False

student_distill = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_FUNSD).to(device)
optimizer_sd = torch.optim.AdamW(student_distill.parameters(), lr=LR, weight_decay=0.01)

print(f"Distillation weights: ALPHA={WEIGHTS.ALPHA_HIDDEN}, BETA={WEIGHTS.BETA_ATTN}, GAMMA={WEIGHTS.GAMMA_LOGITS}, DELTA={WEIGHTS.DELTA_TASK}")

In [ ]:
funsd_distill_results = []
best_f1 = -1.0

for epoch in range(NUM_EPOCHS):
    teacher.eval()
    student_distill.train()
    total_loss = 0.0
    total_distill = 0.0
    total_task = 0.0

    for step, batch in enumerate(funsd_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer_sd.zero_grad()

        model_inputs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"], "bbox": batch["bbox"]}
        teacher_out = teacher(**model_inputs)

        student_hf_out = student_distill.model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
            output_hidden_states=True, output_attentions=True, return_dict=True,
        )
        student_out = {"logits": student_hf_out.logits, "hidden_states": student_hf_out.hidden_states, "attentions": student_hf_out.attentions}

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        distill_loss = distill_losses["loss_total"]
        task_loss = student_hf_out.loss
        loss = distill_loss + WEIGHTS.DELTA_TASK * task_loss

        loss.backward()
        optimizer_sd.step()

        total_loss += loss.item()
        total_distill += distill_loss.item()
        total_task += task_loss.item()

        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(funsd_train_loader)} | total={loss.item():.4f} | distill={distill_loss.item():.4f} | task={task_loss.item():.4f}")

    metrics = evaluate_student(student_distill, funsd_test_loader, device=device)
    funsd_distill_results.append(metrics)
    if metrics["token_f1"] > best_f1:
        best_f1 = metrics["token_f1"]

    print(f"Epoch {epoch+1} | train_loss={total_loss/len(funsd_train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

print(f"\n>>> DocLite (text+layout) Best F1: {best_f1:.4f}")

## 6. FUNSD Results — Text + Layout

In [ ]:
teacher_params = sum(p.numel() for p in teacher_model.parameters())
student_params = sum(p.numel() for p in student_distill.parameters())

t_best = max(funsd_teacher_results, key=lambda x: x['token_f1'])
s_best = max(funsd_student_results, key=lambda x: x['token_f1'])
d_best = max(funsd_distill_results, key=lambda x: x['token_f1'])

print("\n" + "=" * 70)
print("FUNSD RESULTS (Text + Layout)")
print("=" * 70)
print(f"{'Model':<30s} {'Accuracy':>10s} {'F1':>10s} {'Params':>12s}")
print("-" * 70)
print(f"{'LayoutLMv3 (teacher)':<30s} {t_best['token_acc']:>10.4f} {t_best['token_f1']:>10.4f} {teacher_params:>12,}")
print(f"{'LiLT (baseline)':<30s} {s_best['token_acc']:>10.4f} {s_best['token_f1']:>10.4f} {student_params:>12,}")
print(f"{'LiLT + DocLite (ours)':<30s} {d_best['token_acc']:>10.4f} {d_best['token_f1']:>10.4f} {student_params:>12,}")
print("=" * 70)

gap = (d_best['token_f1'] - s_best['token_f1']) / (t_best['token_f1'] - s_best['token_f1'] + 1e-8) * 100
print(f"\nDistillation closed {gap:.1f}% of the teacher-student F1 gap")
print(f"Compression ratio: {teacher_params / student_params:.1f}x fewer parameters")

---
# PART 2: Multimodal Distillation (Teacher sees images)

The teacher (LayoutLMv3) sees **text + layout + document image**.  
The student (LiLT) sees only **text + layout**.  
Through distillation, the student gains image-informed knowledge **without ever seeing images**.

---

In [ ]:
# Reload FUNSD WITH images
TRAIN_IMG_DIR = FUNSD_ROOT / "training_data" / "images"
TEST_IMG_DIR = FUNSD_ROOT / "testing_data" / "images"

print("Loading training data with images...")
funsd_train_img = load_funsd_split(TRAIN_ANN_DIR, tokenizer, image_dir=TRAIN_IMG_DIR)
print("Loading test data with images...")
funsd_test_img = load_funsd_split(TEST_ANN_DIR, tokenizer, image_dir=TEST_IMG_DIR)

funsd_train_img_loader = DataLoader(
    DocumentDataset(funsd_train_img), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_document_batch
)
funsd_test_img_loader = DataLoader(
    DocumentDataset(funsd_test_img), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_document_batch
)

print(f"Has pixel_values: {'pixel_values' in funsd_train_img[0]}")
print(f"pixel_values shape: {torch.tensor(funsd_train_img[0]['pixel_values']).shape}")

## 7. LayoutLMv3 + Image Baseline

In [ ]:
config_img = LayoutLMv3Config.from_pretrained(
    "microsoft/layoutlmv3-base", num_labels=NUM_LABELS_FUNSD,
    output_hidden_states=True, output_attentions=True,
)
teacher_model_img = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base", config=config_img
).to(device)

optimizer_ti = torch.optim.AdamW(teacher_model_img.parameters(), lr=LR, weight_decay=0.01)
eval_wrapper_ti = LayoutLMv3EvalWrapper(teacher_model_img).to(device)

In [ ]:
funsd_teacher_img_results = []

for epoch in range(NUM_EPOCHS):
    teacher_model_img.train()
    total_loss = 0.0

    for step, batch in enumerate(funsd_train_img_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer_ti.zero_grad()

        out = teacher_model_img(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], pixel_values=batch.get("pixel_values"),
            labels=batch["labels"],
        )
        out.loss.backward()
        optimizer_ti.step()
        total_loss += out.loss.item()

        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(funsd_train_img_loader)} | loss={out.loss.item():.4f}")

    metrics = evaluate_student(eval_wrapper_ti, funsd_test_img_loader, device=device)
    funsd_teacher_img_results.append(metrics)
    print(f"Epoch {epoch+1} | train_loss={total_loss/len(funsd_train_img_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

print(f"\n>>> LayoutLMv3+Image Best F1: {max(r['token_f1'] for r in funsd_teacher_img_results):.4f}")

## 8. Multimodal DocLite Distillation

In [ ]:
teacher_mm = TeacherModel("microsoft/layoutlmv3-base", num_labels=NUM_LABELS_FUNSD).to(device)
for param in teacher_mm.parameters():
    param.requires_grad = False

student_mm = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_FUNSD).to(device)
optimizer_mm = torch.optim.AdamW(student_mm.parameters(), lr=LR, weight_decay=0.01)

In [ ]:
funsd_mm_results = []
best_f1_mm = -1.0

for epoch in range(NUM_EPOCHS):
    teacher_mm.eval()
    student_mm.train()
    total_loss = 0.0

    for step, batch in enumerate(funsd_train_img_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer_mm.zero_grad()

        # Teacher sees text + layout + IMAGE
        teacher_inputs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"], "bbox": batch["bbox"]}
        if "pixel_values" in batch:
            teacher_inputs["pixel_values"] = batch["pixel_values"]
        teacher_out = teacher_mm(**teacher_inputs)

        # Student sees ONLY text + layout
        student_hf_out = student_mm.model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
            output_hidden_states=True, output_attentions=True, return_dict=True,
        )
        student_out = {"logits": student_hf_out.logits, "hidden_states": student_hf_out.hidden_states, "attentions": student_hf_out.attentions}

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        loss = distill_losses["loss_total"] + WEIGHTS.DELTA_TASK * student_hf_out.loss

        loss.backward()
        optimizer_mm.step()
        total_loss += loss.item()

        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(funsd_train_img_loader)} | loss={loss.item():.4f}")

    metrics = evaluate_student(student_mm, funsd_test_img_loader, device=device)
    funsd_mm_results.append(metrics)
    if metrics["token_f1"] > best_f1_mm:
        best_f1_mm = metrics["token_f1"]

    print(f"Epoch {epoch+1} | train_loss={total_loss/len(funsd_train_img_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

print(f"\n>>> Multimodal DocLite Best F1: {best_f1_mm:.4f}")

---
# PART 3: SROIE Experiments (Receipts)

Skip this section if you haven't uploaded the SROIE dataset.

---

In [ ]:
from build_sroie_examples import load_sroie_split, NUM_LABELS as NUM_LABELS_SROIE

SROIE_ROOT = ENV.DATA / "sroie"
TRAIN_OCR_DIR = SROIE_ROOT / "train" / "box"
TRAIN_ENT_DIR = SROIE_ROOT / "train" / "entities"
TEST_OCR_DIR = SROIE_ROOT / "test" / "box"
TEST_ENT_DIR = SROIE_ROOT / "test" / "entities"

sroie_train = load_sroie_split(TRAIN_OCR_DIR, TRAIN_ENT_DIR, tokenizer)
sroie_test = load_sroie_split(TEST_OCR_DIR, TEST_ENT_DIR, tokenizer)

sroie_train_loader = DataLoader(
    DocumentDataset(sroie_train), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_document_batch
)
sroie_test_loader = DataLoader(
    DocumentDataset(sroie_test), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_document_batch
)

print(f"SROIE — Train: {len(sroie_train)} receipts | Test: {len(sroie_test)} receipts")

ID2LABEL_SROIE = {0: "company", 1: "date", 2: "address", 3: "total", 4: "other", -100: "[PAD]"}
all_labels_s = [l for ex in sroie_train for l in ex["labels"] if l != -100]
counts_s = Counter(all_labels_s)
print("\nLabel distribution (training):")
for lid, c in sorted(counts_s.items()):
    print(f"  {ID2LABEL_SROIE[lid]:>10s}: {c:>5d} ({c/len(all_labels_s)*100:.1f}%)")

## 9. SROIE: Teacher Baseline

In [ ]:
config_s = LayoutLMv3Config.from_pretrained(
    "microsoft/layoutlmv3-base", num_labels=NUM_LABELS_SROIE,
    output_hidden_states=True, output_attentions=True,
)
sroie_teacher = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base", config=config_s
).to(device)
optimizer_st = torch.optim.AdamW(sroie_teacher.parameters(), lr=LR, weight_decay=0.01)
eval_wrapper_st = LayoutLMv3EvalWrapper(sroie_teacher).to(device)

sroie_teacher_results = []

for epoch in range(NUM_EPOCHS):
    sroie_teacher.train()
    total_loss = 0.0

    for step, batch in enumerate(sroie_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer_st.zero_grad()
        out = sroie_teacher(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer_st.step()
        total_loss += out.loss.item()

    metrics = evaluate_student(eval_wrapper_st, sroie_test_loader, device=device)
    sroie_teacher_results.append(metrics)
    print(f"Epoch {epoch+1} | train_loss={total_loss/len(sroie_train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

print(f"\n>>> SROIE LayoutLMv3 Best F1: {max(r['token_f1'] for r in sroie_teacher_results):.4f}")

## 10. SROIE: Student Baseline

In [ ]:
sroie_student = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_SROIE).to(device)
optimizer_ss = torch.optim.AdamW(sroie_student.parameters(), lr=LR, weight_decay=0.01)

sroie_student_results = []

for epoch in range(NUM_EPOCHS):
    sroie_student.train()
    total_loss = 0.0

    for step, batch in enumerate(sroie_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer_ss.zero_grad()
        out = sroie_student.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer_ss.step()
        total_loss += out.loss.item()

    metrics = evaluate_student(sroie_student, sroie_test_loader, device=device)
    sroie_student_results.append(metrics)
    print(f"Epoch {epoch+1} | train_loss={total_loss/len(sroie_train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

print(f"\n>>> SROIE LiLT Baseline Best F1: {max(r['token_f1'] for r in sroie_student_results):.4f}")

## 11. SROIE: DocLite Distillation

In [ ]:
sroie_teacher_d = TeacherModel("microsoft/layoutlmv3-base", num_labels=NUM_LABELS_SROIE).to(device)
for param in sroie_teacher_d.parameters():
    param.requires_grad = False

sroie_student_d = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_SROIE).to(device)
optimizer_ssd = torch.optim.AdamW(sroie_student_d.parameters(), lr=LR, weight_decay=0.01)

sroie_distill_results = []
best_f1_sroie = -1.0

for epoch in range(NUM_EPOCHS):
    sroie_teacher_d.eval()
    sroie_student_d.train()
    total_loss = 0.0

    for step, batch in enumerate(sroie_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer_ssd.zero_grad()

        model_inputs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"], "bbox": batch["bbox"]}
        teacher_out = sroie_teacher_d(**model_inputs)

        student_hf_out = sroie_student_d.model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
            output_hidden_states=True, output_attentions=True, return_dict=True,
        )
        student_out = {"logits": student_hf_out.logits, "hidden_states": student_hf_out.hidden_states, "attentions": student_hf_out.attentions}

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        loss = distill_losses["loss_total"] + WEIGHTS.DELTA_TASK * student_hf_out.loss

        loss.backward()
        optimizer_ssd.step()
        total_loss += loss.item()

    metrics = evaluate_student(sroie_student_d, sroie_test_loader, device=device)
    sroie_distill_results.append(metrics)
    if metrics["token_f1"] > best_f1_sroie:
        best_f1_sroie = metrics["token_f1"]

    print(f"Epoch {epoch+1} | train_loss={total_loss/len(sroie_train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

print(f"\n>>> SROIE DocLite Best F1: {best_f1_sroie:.4f}")

## 12. SROIE Results

In [ ]:
st_best = max(sroie_teacher_results, key=lambda x: x['token_f1'])
ss_best = max(sroie_student_results, key=lambda x: x['token_f1'])
sd_best = max(sroie_distill_results, key=lambda x: x['token_f1'])

print("\n" + "=" * 70)
print("SROIE RESULTS")
print("=" * 70)
print(f"{'Model':<30s} {'Accuracy':>10s} {'F1':>10s}")
print("-" * 70)
print(f"{'LayoutLMv3 (teacher)':<30s} {st_best['token_acc']:>10.4f} {st_best['token_f1']:>10.4f}")
print(f"{'LiLT (baseline)':<30s} {ss_best['token_acc']:>10.4f} {ss_best['token_f1']:>10.4f}")
print(f"{'LiLT + DocLite (ours)':<30s} {sd_best['token_acc']:>10.4f} {sd_best['token_f1']:>10.4f}")
print("=" * 70)

gap_s = (sd_best['token_f1'] - ss_best['token_f1']) / (st_best['token_f1'] - ss_best['token_f1'] + 1e-8) * 100
print(f"\nDistillation closed {gap_s:.1f}% of the teacher-student F1 gap")

---
# PART 4: Final Summary
---

In [ ]:
print("\n" + "=" * 80)
print("COMPLETE RESULTS SUMMARY")
print("=" * 80)

# FUNSD
print(f"\n{'FUNSD (Forms)':^80s}")
print("-" * 80)
print(f"{'Model':<35s} {'Inputs':<12s} {'Acc':>8s} {'F1':>8s} {'Params':>12s}")
print("-" * 80)
print(f"{'LayoutLMv3 (teacher)':<35s} {'T+L':<12s} {t_best['token_acc']:>8.4f} {t_best['token_f1']:>8.4f} {teacher_params:>12,}")
print(f"{'LiLT (baseline)':<35s} {'T+L':<12s} {s_best['token_acc']:>8.4f} {s_best['token_f1']:>8.4f} {student_params:>12,}")
print(f"{'LiLT + DocLite (ours)':<35s} {'T+L':<12s} {d_best['token_acc']:>8.4f} {d_best['token_f1']:>8.4f} {student_params:>12,}")

ti_best = max(funsd_teacher_img_results, key=lambda x: x['token_f1'])
mm_best = max(funsd_mm_results, key=lambda x: x['token_f1'])
print(f"{'LayoutLMv3 + Image (teacher)':<35s} {'T+L+I':<12s} {ti_best['token_acc']:>8.4f} {ti_best['token_f1']:>8.4f} {teacher_params:>12,}")
print(f"{'LiLT + DocLite multimodal':<35s} {'T+L':<12s} {mm_best['token_acc']:>8.4f} {mm_best['token_f1']:>8.4f} {student_params:>12,}")

print(f"\n{'SROIE (Receipts)':^80s}")
print("-" * 80)
print(f"{'Model':<35s} {'Inputs':<12s} {'Acc':>8s} {'F1':>8s}")
print("-" * 80)
print(f"{'LayoutLMv3 (teacher)':<35s} {'T+L':<12s} {st_best['token_acc']:>8.4f} {st_best['token_f1']:>8.4f}")
print(f"{'LiLT (baseline)':<35s} {'T+L':<12s} {ss_best['token_acc']:>8.4f} {ss_best['token_f1']:>8.4f}")
print(f"{'LiLT + DocLite (ours)':<35s} {'T+L':<12s} {sd_best['token_acc']:>8.4f} {sd_best['token_f1']:>8.4f}")
print("=" * 80)

print(f"\nKey: T=Text, L=Layout, I=Image")
print(f"Compression: {teacher_params/student_params:.1f}x fewer parameters")
print(f"Note: Multimodal student NEVER sees images — it learns visual knowledge through distillation.")